In [0]:
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("domain_name", "")
dbutils.widgets.text("sub_domain_name", "")

table_name = dbutils.widgets.get("table_name").strip()
domain_name = dbutils.widgets.get("domain_name").strip()
sub_domain_name = dbutils.widgets.get("sub_domain_name").strip()

In [0]:
def trigger_alert(event_type, execution_id=None, asset_id=None, message=None):
    """
    Placeholder for future alerting.

    Future implementation could:
    - send email
    - create UI notification
    - trigger webhook
    - notify the requesting user
    """
    pass

In [0]:
from pyspark.sql import functions as F

CATALOG = "mastercard_card_241614"
SCHEMA = "default"

assets_df = spark.table(
    f"{CATALOG}.{SCHEMA}.assets_m"
)

domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.domains_m"
)

sub_domains_df = spark.table(
    f"{CATALOG}.{SCHEMA}.sub_domains_m"
)

In [0]:
asset_df = (
    assets_df
    .filter(
        F.lower(F.col("asset_name")) == table_name.lower()
    )
)

In [0]:
asset_count = asset_df.count()

if asset_count == 0:
    message = (
        f"No asset found for table='{table_name}', "
        f"domain='{domain_name}', "
        f"sub_domain='{sub_domain_name}'"
    )

    trigger_alert(
        event_type="ASSET_NOT_FOUND",
        message=message
    )

    raise Exception(message)

if asset_count > 1:
    message = f"Multiple assets found for table='{table_name}'"

    trigger_alert(
        event_type="MULTIPLE_ASSETS_FOUND",
        message=message
    )

    raise Exception(message)

asset = asset_df.first()

asset_id = asset.asset_id
asset_domain_id = asset.domain_id
asset_sub_domain_id = asset.sub_domain_id

In [0]:
domain_df = (
    domains_df
    .filter(
        F.col("domain_id") == asset_domain_id
    )
)

domain = domain_df.first()

if domain is None:
    message = (
        f"Domain metadata missing for asset_id={asset_id}"
    )

    trigger_alert(
        event_type="DOMAIN_METADATA_MISSING",
        asset_id=asset_id,
        message=message
    )

    raise Exception(message)

In [0]:
sub_domain_df = (
    sub_domains_df
    .filter(
        F.col("sub_domain_id") == asset_sub_domain_id
    )
)

sub_domain = sub_domain_df.first()

if sub_domain is None:
    message = (
        f"Sub-domain metadata missing for asset_id={asset_id}"
    )

    trigger_alert(
        event_type="SUB_DOMAIN_METADATA_MISSING",
        asset_id=asset_id,
        message=message
    )

    raise Exception(message)

In [0]:
# if (
#     sub_domain.sub_domain_name.lower() != sub_domain_name.lower()
#     or sub_domain.domain_id != asset_domain_id
# ):

#     message = (
#         f"Sub-domain mismatch for table='{table_name}'. "
#         f"Expected='{sub_domain.sub_domain_name}' "
#         f"under domain_id={asset_domain_id}, "
#         f"received='{sub_domain_name}'"
#     )

#     trigger_alert(
#         event_type="SUB_DOMAIN_MISMATCH",
#         asset_id=asset_id,
#         message=message
#     )

#     raise Exception(message)

In [0]:
print(
    f"✅ Asset validation passed: "
    f"{domain.domain_name} / "
    f"{sub_domain.sub_domain_name} / "
    f"{table_name}"
)

print(f"Asset ID: {asset_id}")

✅ Asset validation passed: TXN_PROCESSING / TRANSACTION_LIFECYCLE_STATUS / 01_TXN_transaction_lifecycle
Asset ID: 28


In [0]:
import pandas as pd

EXCEL_PATH = (
    "/Volumes/mastercard_card_241614/default/"
    "test_data/PCN_DQ_TestDataset (1).xlsx"
)

pdf = pd.read_excel(
    EXCEL_PATH,
    sheet_name=table_name
)

In [0]:
for col in pdf.select_dtypes(include=['object']).columns:
    pdf[col] = pdf[col].where(pdf[col].isna(), pdf[col].astype(str))

df = spark.createDataFrame(pdf)

In [0]:
expected_df = (
    spark.table(
        f"{CATALOG}.{SCHEMA}.asset_columns_m"
    )
    .filter(
        F.col("asset_id") == asset_id
    )
    .orderBy("ordinal_position")
)

In [0]:
expected_columns = [
    row.column_name
    for row in expected_df.select("column_name").collect()
]

actual_columns = df.columns

In [0]:
expected_set = {c.lower() for c in expected_columns}
actual_set = {c.lower() for c in actual_columns}

missing_columns = sorted(
    expected_set - actual_set
)

unexpected_columns = sorted(
    actual_set - expected_set
)

In [0]:
if missing_columns or unexpected_columns:

    message = (
        f"Schema validation failed for asset_id={asset_id}. "
        f"Missing columns={missing_columns}, "
        f"Unexpected columns={unexpected_columns}"
    )

    trigger_alert(
        event_type="SCHEMA_VALIDATION_FAILED",
        asset_id=asset_id,
        message=message
    )

    raise Exception(message)

---------------------------------------------------------------------------
Exception                                 Traceback (most recent call last)
File <command-4609406856124558>, line 15
      3 message = (
      4     f"Schema validation failed for asset_id={asset_id}. "
      5     f"Missing columns={missing_columns}, "
      6     f"Unexpected columns={unexpected_columns}"
      7 )
      9 trigger_alert(
     10     event_type="SCHEMA_VALIDATION_FAILED",
     11     asset_id=asset_id,
     12     message=message
     13 )
---> 15 raise Exception(message)

Exception: Schema validation failed for asset_id=28. Missing columns=['account_id', 'changed_by', 'effective_date', 'reason_code', 'status', 'status_id'], Unexpected columns=['amount', 'authorized_at', 'channel', 'cleared_at', 'currency_code', 'initiated_at', 'is_cross_border', 'lifecycle_stage', 'network', 'pan_last4', 'settled_at', 'transaction_id', 'transaction_type']

In [0]:
print(
    f"✅ Schema validation passed for "
    f"{domain_name}.{sub_domain_name}.{table_name}"
)

In [0]:
dbutils.notebook.exit(
    "SCHEMA_VALIDATION_PASSED"
)